# Train Fixes — Revive ADMET and Toxic Colors

> **Status:** Not yet run on Colab. Unverified.

The original ADMET (Shi 2019) and Toxic Colors (Fernandez 2018) architectures
collapsed to constant predictions. Both have large first-layer convs (21x21,
16x16) that are prone to dead ReLU with default Glorot init on [0,1] inputs.

**Fixes applied:**
1. `he_normal` kernel initialization (variance scaled for ReLU)
2. `LeakyReLU(0.1)` instead of `ReLU` (neurons can't fully die)
3. `BatchNormalization` after the first conv (stabilizes large-kernel activations)
4. Target standardization — train on z-scored pIC50, invert for reporting
5. `Adam(learning_rate=1e-4)` (gentler updates for large kernels)

`IMG_SIZE` defaults to 180 to match the originals. The papers used 300×300;
bump `IMG_SIZE = (300, 300)` if Colab has the memory (~2.8× compute).

Same data split as `train_and_save.ipynb`. Saves results to `results/`
alongside the originals for comparison.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/Molecule_CV'

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_absolute_error

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## Load Data (same pipeline as train_and_save)

In [ ]:
df = pd.read_csv(os.path.join(DATA_DIR, 'bace.csv'))
IMG_DIR = os.path.join(DATA_DIR, 'molecule_images')

IMG_SIZE = (180, 180)  # papers used (300, 300); bump if compute allows

def load_images_and_targets(df, img_dir, img_size=IMG_SIZE):
    images, pic50s, classes, cids = [], [], [], []
    missing = 0
    for _, row in df.iterrows():
        img_path = os.path.join(img_dir, f"{row['CID']}.png")
        if not os.path.exists(img_path):
            missing += 1
            continue
        img = tf.io.read_file(img_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32) / 255.0
        images.append(img)
        pic50s.append(row['pIC50'])
        classes.append(row['Class'])
        cids.append(row['CID'])
    if missing:
        print(f"Warning: {missing} images not found")
    return np.array(images), np.array(pic50s), np.array(classes), cids

images, pic50, labels, cids = load_images_and_targets(df, IMG_DIR)
print(f"Loaded {len(images)} images at {IMG_SIZE}")

X_temp, X_test, y_temp, y_test, c_temp, c_test, cid_temp, cid_test = train_test_split(
    images, pic50, labels, cids, test_size=0.2, random_state=SEED)
X_train, X_val, y_train, y_val, c_train, c_val, cid_train, cid_val = train_test_split(
    X_temp, y_temp, c_temp, cid_temp, test_size=0.5, random_state=SEED)

# Standardize target using train stats only (no leakage)
Y_MEAN = float(y_train.mean())
Y_STD = float(y_train.std())
y_train_z = (y_train - Y_MEAN) / Y_STD
y_val_z = (y_val - Y_MEAN) / Y_STD
y_test_z = (y_test - Y_MEAN) / Y_STD

print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
print(f"pIC50 train stats — mean: {Y_MEAN:.3f}, std: {Y_STD:.3f}")

## Fixed Architectures

Same layer stacks as the originals, with these changes:
- `kernel_initializer='he_normal'` on all Conv2D/Dense layers
- `LeakyReLU(0.1)` replacing `activation='relu'`
- `BatchNormalization` after the first Conv2D (before activation)
- Trained with `Adam(learning_rate=1e-4)` on z-scored pIC50

In [ ]:
def build_admet_fixed(input_shape=IMG_SIZE + (3,)):
    """Shi et al. 2019 — he_normal init, BatchNorm, LeakyReLU"""
    return keras.Sequential([
        keras.layers.Conv2D(16, (21, 21), kernel_initializer='he_normal',
                            input_shape=input_shape),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(0.1),
        keras.layers.MaxPooling2D((14, 14), padding='same'),
        keras.layers.Flatten(),
        keras.layers.Dense(512, kernel_initializer='he_normal'),
        keras.layers.LeakyReLU(0.1),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1)
    ])

def build_toxic_colors_fixed(input_shape=IMG_SIZE + (3,)):
    """Fernandez et al. 2018 — he_normal init, BatchNorm, LeakyReLU"""
    return keras.Sequential([
        keras.layers.Conv2D(12, (16, 16), kernel_initializer='he_normal',
                            input_shape=input_shape),
        keras.layers.BatchNormalization(),
        keras.layers.LeakyReLU(0.1),
        keras.layers.Dropout(0.4),
        keras.layers.MaxPooling2D((2, 2)),
        keras.layers.Flatten(),
        keras.layers.Dense(40, kernel_initializer='he_normal'),
        keras.layers.LeakyReLU(0.1),
        keras.layers.Dropout(0.4),
        keras.layers.Dense(1)
    ])

MODELS = {
    'ADMET Fixed': build_admet_fixed,
    'Toxic Colors Fixed': build_toxic_colors_fixed,
}

## Train

In [ ]:
results = {}

for name, builder in MODELS.items():
    print(f"\n{'='*50}")
    print(f"Training: {name}")
    print(f"{'='*50}")

    model = builder()
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-4),
        loss='mse',
        metrics=['mae'],
    )

    # Train on z-scored target
    history = model.fit(
        X_train, y_train_z,
        validation_data=(X_val, y_val_z),
        epochs=100,
        batch_size=128,
        verbose=1,
    )

    # Predict in z-space, then invert to pIC50 scale
    preds_z = model.predict(X_test, verbose=0).flatten()
    preds = preds_z * Y_STD + Y_MEAN

    # Metrics on original pIC50 scale
    test_mae = float(np.mean(np.abs(preds - y_test)))
    test_mse = float(np.mean((preds - y_test) ** 2))

    results[name] = {
        'history': history.history,
        'predictions': preds,
        'test_loss': test_mse,
        'test_mae': test_mae,
    }
    print(f"\n{name} — Test MSE: {test_mse:.4f}, Test MAE: {test_mae:.4f}")
    print(f"  Prediction range: {preds.min():.4f} to {preds.max():.4f}")
    print(f"  Unique predictions: {len(np.unique(preds.round(4)))}")

## Sanity Check — Did They Actually Learn?

If unique predictions > 1 and the range spans more than a sliver of
the pIC50 range (3.1 to 9.2), the fix worked.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, len(MODELS), figsize=(6 * len(MODELS), 5))
if len(MODELS) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, results.items()):
    preds = res['predictions']
    ax.scatter(y_test, preds, alpha=0.5, s=15)
    lo = min(y_test.min(), preds.min()) - 0.5
    hi = max(y_test.max(), preds.max()) + 0.5
    ax.plot([lo, hi], [lo, hi], 'k--', alpha=0.3)
    ax.set_title(f"{name}\nMAE={res['test_mae']:.3f}, unique={len(np.unique(preds.round(4)))}")
    ax.set_xlabel('Actual pIC50')
    ax.set_ylabel('Predicted pIC50')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
fig.tight_layout()
plt.show()

## Save Results

In [ ]:
OUTPUT_DIR = os.path.join(DATA_DIR, 'results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Predictions — same test set as train_and_save, so CIDs align
pred_df = pd.DataFrame({'CID': cid_test, 'pIC50_actual': y_test, 'Class': c_test})
for name, res in results.items():
    col = name.lower().replace(' ', '_')
    pred_df[f'pred_{col}'] = res['predictions']
pred_df.to_csv(os.path.join(OUTPUT_DIR, 'predictions_fixed.csv'), index=False)
print(f"Saved predictions_fixed.csv ({len(pred_df)} rows)")

# Metrics
metrics_rows = []
for name, res in results.items():
    auc = roc_auc_score(c_test, res['predictions'])
    metrics_rows.append({
        'model': name,
        'test_mse': res['test_loss'],
        'test_mae': res['test_mae'],
        'auc_roc': round(auc, 4),
    })
metrics_df = pd.DataFrame(metrics_rows)
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'model_metrics_fixed.csv'), index=False)
print("Saved model_metrics_fixed.csv")
print(metrics_df.to_string(index=False))

# Histories
for name, res in results.items():
    col = name.lower().replace(' ', '_')
    hist_df = pd.DataFrame(res['history'])
    hist_df.index.name = 'epoch'
    hist_df.to_csv(os.path.join(OUTPUT_DIR, f'history_{col}.csv'))
print("Saved training histories")

In [ ]:
import shutil
from google.colab import files

zip_base = '/content/molecule_cv_results_fixed'
shutil.make_archive(zip_base, 'zip', OUTPUT_DIR)
print(f'Zipped {OUTPUT_DIR} -> {zip_base}.zip')
files.download(f'{zip_base}.zip')